In [1]:
# 8B baseline with 4bit quantization

import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import gc
from unsloth import FastLanguageModel

import evaluate
import numpy as np

gc.collect()
torch.cuda.empty_cache()
print("Booting up Baseline Agentic Pipeline (Zero Adapters)...")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Booting up Baseline Agentic Pipeline (Zero Adapters)...


In [2]:
# untrained 8B Base Model
BASE_MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"


print(f"Loading Base Model: {BASE_MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_NAME, 
    max_seq_length = 2048, 
    dtype = None,
    load_in_4bit = True, # for 8B model on an 8GB GPU!
)
FastLanguageModel.for_inference(model)
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

print("Skipping Adapters............ Running the llama 8B 4bit Baseline Model.")

Loading Base Model: unsloth/Meta-Llama-3.1-8B-bnb-4bit...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Skipping Adapters............ Running the llama 8B 4bit Baseline Model.


In [3]:
# Master PROMPT
TASK_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{document_text}

### Response:\n"""

In [4]:
# hyperparameter Dictionary
TOOL_CONFIGS = {
    "extract_invoice": {
        "instruction": "Extract ALL key-value pairs from the invoice. Output ONLY valid JSON.",
        "max_tokens": 800,
        "temperature": 0.1,        
        "repetition_penalty": 1.1 
    },
    "summarize_meeting": {
        "instruction": "Summarize the transcript. Extract key decisions and action items. Keep the summary concise. DO NOT add external information or hallucinate goals.",
        "max_tokens": 150,         
        "temperature": 0.1,        
        "repetition_penalty": 1.2  
    },
    "read_form": {
        "instruction": "Extract key-value pairs from the unstructured form. Output a 'Thought' summarizing the layout, then the JSON.",
        "max_tokens": 800,
        "temperature": 0.1,        
        "repetition_penalty": 1.1  
    },
    "classify_email": {
        "instruction": "Determine the sender's core intent. Extract ALL required action items. Output a 'Thought' followed by the 'Action'.",
        "max_tokens": 300,
        "temperature": 0.1,
        "repetition_penalty": 1.1
    }
}

In [5]:
def find_document_structure(document_text):
    """Deterministic Rule-Based Router: Scans the actual document text for structural clues."""
    doc_lower = document_text.lower()
    
    # Look for standard email headers
    if document_text.strip().startswith("Subject:") or ("to:" in doc_lower[:100] and "from:" in doc_lower[:100]):
        return "classify_email"
        
    # look for financial math keywords
    elif "subtotal" in doc_lower or "tax" in doc_lower or "total:" in doc_lower:
        return "extract_invoice"
        
    # look for form elements)
    elif "dob:" in doc_lower or "registration" in doc_lower or "application" in doc_lower or "gender:" in doc_lower:
        return "read_form"
        
    # look for Speaker patterns
    lines = document_text.split('\n')
    speaker_lines = sum(1 for line in lines[:5] if ":" in line and len(line.split(":")[0]) < 15)
    if speaker_lines >= 2 and not document_text.strip().startswith("Subject:"):
        return "summarize_meeting"
        
    return "unknown"

In [6]:
def process_user_request(user_query, document_content):
    print("\n" + "="*70)
    print(f"USER REQUEST: {user_query}")
    print("="*70)

    print("Finding document structure...")
    target_tool = find_document_structure(document_content)
    
    if target_tool != "unknown":
        print(f"Found structural match: {target_tool.upper()}")

    else:
        print("Structure unclear. Activating AI Router...")
        model.disable_adapters() # Turn off skills, use raw Llama-3 brain
        
        ROUTING_PROMPT = f"""Analyze the user's request and select the exact tool needed. 
Tools available:
- extract_invoice
- summarize_meeting
- read_form
- classify_email

User Request: "{user_query}"

Respond with ONLY the exact name of the tool from the list above. Do not output any other text.
Tool Name:"""

        inputs = tokenizer([ROUTING_PROMPT], return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs, 
            max_new_tokens=25, 
            temperature=0.1, 
            use_cache=True, 
            eos_token_id=terminators
        )
        
        raw_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        try:
            raw_tool_string = raw_output.split("Tool Name:")[1].strip().lower()
        except IndexError:
            raw_tool_string = ""
            
        target_tool = "unknown"
        for tool in TOOL_CONFIGS.keys():
            if tool in raw_tool_string:
                target_tool = tool
                break
                
        print(f"BASE MODEL OUTPUT: {target_tool if target_tool != 'unknown' else raw_tool_string}")

        if target_tool == "unknown":
            print("WARNING: Base model routing failed. Engaging keyword safety net...")
            query_lower = user_query.lower()
            if any(w in query_lower for w in ["email", "message", "update", "intent", "action item"]): target_tool = "classify_email"
            elif any(w in query_lower for w in ["receipt", "invoice", "bill", "total"]): target_tool = "extract_invoice"
            elif any(w in query_lower for w in ["form", "application"]): target_tool = "read_form"
            elif any(w in query_lower for w in ["meeting", "transcript", "sync", "audio"]): target_tool = "summarize_meeting"
            else:
                print("Could not route task. Aborting.")
                return

    print(f"\nRouting task to adapter: {target_tool.upper()} <-")

    # no adapters
    print(f"Using PURE BASELINE for {target_tool} task...")
    
    # pulls exac parameter from dictionary
    config = TOOL_CONFIGS[target_tool]
    task_instruction = config["instruction"]
    gen_tokens = config["max_tokens"]
    gen_temp = config["temperature"]
    gen_rep_penalty = config["repetition_penalty"]
    
    execution_prompt = TASK_PROMPT.format(instruction=task_instruction, document_text=document_content)
    inputs = tokenizer([execution_prompt], return_tensors="pt").to("cuda")
    print(f"Processing document...")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=gen_tokens, 
        temperature=gen_temp, 
        use_cache=True, 
        repetition_penalty=gen_rep_penalty, 
        eos_token_id=terminators
    )
    
    final_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("### Response:\n")[1].strip()
    
    print("\n" + "*"*70)
    print("FINAL OUTPUT")
    print("*"*70)
    print(final_output)
    print("*"*70 + "\n")

    return final_output

In [7]:
# Pipline test 
print("\nLoading BERTScore for Live Evaluation...")
bertscore = evaluate.load("bertscore")

test_query_1 = "I found this old receipt from Home Depot, can you get the total amount and company name?"
test_document_1 = "HOME DEPOT #4451\n123 BUILDER BLVD\n\n1x HAMMER - $15.99\n2x NAIL PACK - $4.50\n\nSUBTOTAL: $24.99\nTAX: $2.00\nTOTAL: $26.99\nTHANK YOU FOR SHOPPING!"
gold_1 = '{"merchant": "HOME DEPOT", "merchant_id": "4451", "address": "123 BUILDER BLVD", "total": "26.99"}'

test_query_2 = "Look at this patient intake form and pull out the demographic info."
test_document_2 = "PATIENT REGISTRATION FORM\nName: Jane Doe\nDOB: 05/14/1985\nGender: F\nPhone: (555) 123-4567\nEmergency Contact: John Doe\nRelation: Spouse"
gold_2 = 'Thought: The form contains patient registration details including demographic and emergency contact information.\nJSON: {"Name": "Jane Doe", "DOB": "05/14/1985", "Gender": "F", "Phone": "(555) 123-4567", "Emergency Contact": "John Doe", "Relation": "Spouse"}'

test_query_3 = "I missed the design sync this morning. What did the team decide in this transcript?"
test_document_3 = "Alice: So, are we going with the blue theme or the green theme for the app launch?\nBob: Marketing prefers the blue theme. It tested better with the focus groups.\nCharlie: Agreed. Let's lock in the blue theme.\nAlice: Perfect. Bob, can you update the Figma files by Wednesday?\nBob: Will do."
gold_3 = "Summary: The team decided to use the blue theme for the app launch based on marketing preferences.\nKey decisions: \n- Proceed with the blue theme for the app launch.\nAction items: \n- Bob to update the Figma files by Wednesday."

test_query_4 = "Sarah just sent this long project update, what is the main intent and do I have any action items?"
test_document_4 = "Subject: Q1 Campaign Launch & Asset Review\n\nHi everyone,\n\nJust a quick update that the Q1 marketing campaign is officially live! Great work across the board.\n\nHowever, we noticed a minor typo in the banner graphics on the landing page. David, please review the uploaded assets in the shared drive and push the corrected versions to production by EOD today so we don't lose click-throughs over the weekend.\n\nBest,\nSarah"
gold_4 = "Thought: The sender is announcing the launch of the Q1 campaign but also flagging an urgent typo in the banner graphics that needs fixing today.\nAction: David needs to review the uploaded assets in the shared drive and push the corrected versions to production by EOD today."

test_query_5 = "Richard sent a super passive-aggressive message about Q3, what does he actually want us to do?"
test_document_5 = "Subject: URGENT: Q3 Financial Reports & Friday Sync\n\nHi Team,\n\nI need everyone to focus. Bob, can you please pull the final Q3 revenue numbers and send them to my inbox by EOD tomorrow? We are presenting to the board on Monday.\n\nAlso, Susan, I think we need to align on the upcoming merger integration timeline. Let's schedule a 15-minute sync for this Friday at 2:00 PM to hammer out the details.\n\nThanks,\nRichard"
gold_5 = "Thought: The sender is urgently requesting Q3 financial data for a board presentation and wants to set up a meeting to discuss merger integration.\nAction: \n- Bob: Pull final Q3 revenue numbers and send them to Richard's inbox by EOD tomorrow.\n- Susan: Schedule a 15-minute sync for this Friday at 2:00 PM."



Loading BERTScore for Live Evaluation...


In [8]:
predictions = []
references = [gold_1, gold_2, gold_3, gold_4, gold_5]

print("\n\n" + "#"*70)
print("RUNNING LIVE TESTS ( on 8B BASELINE no adapters)")
print("#"*70)

predictions.append(process_user_request(test_query_1, test_document_1))
predictions.append(process_user_request(test_query_2, test_document_2))
predictions.append(process_user_request(test_query_3, test_document_3))
predictions.append(process_user_request(test_query_4, test_document_4))
predictions.append(process_user_request(test_query_5, test_document_5))



######################################################################
RUNNING LIVE TESTS ( on 8B BASELINE no adapters)
######################################################################

USER REQUEST: I found this old receipt from Home Depot, can you get the total amount and company name?
Finding document structure...
Found structural match: EXTRACT_INVOICE

Routing task to adapter: EXTRACT_INVOICE <-
Using PURE BASELINE for extract_invoice task...
Processing document...

**********************************************************************
FINAL OUTPUT
**********************************************************************
{
"Hammer": "$15.99",
"Nail Pack": "$4.50"
}
**********************************************************************


USER REQUEST: Look at this patient intake form and pull out the demographic info.
Finding document structure...
Found structural match: READ_FORM

Routing task to adapter: READ_FORM <-
Using PURE BASELINE for read_form task...
Processing docum

In [9]:
print("\n" + "="*70)
print("CALCULATING PIPELINE BERTSCORE (baseline 8B no adapters with 4bit quantization)...")
print("="*70)

# Calculate scores
scores = bertscore.compute(predictions=predictions, references=references, lang="en")
f1_scores = scores['f1']

print(f"Test 1 F1: {f1_scores[0]*100:.2f}%")
print(f"Test 2 F1: {f1_scores[1]*100:.2f}%")
print(f"Test 3 F1: {f1_scores[2]*100:.2f}%")
print(f"Test 4 F1: {f1_scores[3]*100:.2f}%")
print(f"Test 5 F1: {f1_scores[4]*100:.2f}%")
print("-" * 50)
print(f"OVERALL PIPELINE SCORE: {np.mean(f1_scores)*100:.2f}%")
print("="*70)


CALCULATING PIPELINE BERTSCORE (baseline 8B no adapters with 4bit quantization)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test 1 F1: 83.47%
Test 2 F1: 85.33%
Test 3 F1: 88.32%
Test 4 F1: 91.28%
Test 5 F1: 84.88%
--------------------------------------------------
OVERALL PIPELINE SCORE: 86.66%


In [10]:
# HARD SHUTDOWN & UNLOAD FROM GPU

import torch
import gc

print("Initiating VRAM Hard-Shutdown for Local Models...")

# 1. Track memory before cleanup
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated Before: {vram_before:.2f} GB")

# 2. List of every heavy object that might be trapped in memory
heavy_objects = [
    'model', 'tokenizer', 'trainer', 'bertscore', 'rouge', 
    'dataset', 'train_dataset', 'eval_dataset', 'split_dataset',
    'inputs', 'outputs'
]

# 3. Delete them dynamically if they exist
for obj in heavy_objects:
    if obj in globals():
        print(f"Unloading '{obj}' from memory...")
        del globals()[obj]

# 4. Force aggressive Garbage Collection (Run twice to clear circular references)
gc.collect()
gc.collect()

# 5. Flush the GPU Cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Clears memory shared between backend processes
    torch.cuda.synchronize()  # Waits for all GPU operations to completely finish
    
    # Track memory after cleanup
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated After:  {vram_after:.2f} GB")

print("\nGPU Memory Cleared. Your VRAM is now completely empty!")

Initiating VRAM Hard-Shutdown for Local Models...
VRAM Allocated Before: 6.46 GB
Unloading 'model' from memory...
Unloading 'tokenizer' from memory...
Unloading 'bertscore' from memory...
VRAM Allocated After:  0.01 GB

GPU Memory Cleared. Your VRAM is now completely empty!
